# RAG 기본 프로세스

### 라이브러리 선언

In [ ]:
import os
from dotenv import load_dotenv

# 환경 변수 로드 (.env)
load_dotenv()

# RAG 프로세스 핵심 라이브러리
import langchain, chromadb, pypdf
# PDF 로더
from langchain_community.document_loaders import PyPDFLoader
# 텍스트 청크 분할
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 임베딩모델 OpenAI 라이브러리
from langchain_openai import OpenAIEmbeddings
# 임베딩모델 무료 라이브러리 (허깅페이스)
from langchain_community.embeddings import HuggingFaceEmbeddings
# 벡터 DB 생성 라이브러리
from langchain_community.vectorstores import Chroma
# 모델 GPT 사용 라이브러리
from langchain_openai import ChatOpenAI
# 모델 GGUF 사용 라이브러리
from langchain_community.llms import LlamaCpp
# 모델 OLLAMA 사용 라이브러리
from langchain_community.llms import Ollama
# 임베딩모델 Ollama 라이브러리
from langchain_community.embeddings import OllamaEmbeddings
# 프롬프트 템플릿 라이브러리
from langchain_core.prompts import PromptTemplate
# RAG 체인 구성 라이브러리
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

### 환경 설정

In [ ]:
# 폴더 구조 설정
DATA_DIR = "dataset"
PDF_PATH = os.path.join(DATA_DIR, "모집요강.pdf")
PERSIST_DIR = "vector_db"
MODEL_DIR = "models"
OPENAI_APIKEY = os.getenv("OPENAI_API_KEY")

# LLM 모델 선택: "gpt" 또는 "ollama"
MODEL_TYPE = "ollama"   # "gpt" 또는 "ollama" 중 선택
OLLAMA_MODEL = "gemma4:e2b"  # Ollama 사용 시 모델명

# 폴더 생성
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(PERSIST_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("데이터 폴더:", DATA_DIR)
print("PDF 경로:", PDF_PATH)
print("저장 경로:", PERSIST_DIR)

In [ ]:
# 설정값 정의
CHUNK_SIZE = 800        # 청크 크기 (권장: 600-1200)
CHUNK_OVERLAP = 150     # 청크 오버랩 (권장: 100-200)
K_RETRIEVAL = 4         # 검색할 문서 수 (권장: 3-6)

print("설정된 하이퍼파라미터:")
print(f"청크 크기: {CHUNK_SIZE}")
print(f"오버랩: {CHUNK_OVERLAP}")
print(f"검색 수: {K_RETRIEVAL}")

# 1. 데이터 준비

### 1-1. Data Load

In [ ]:
# PDF 로딩 실행
loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

print(f" {len(docs)}개 페이지 로딩 완료!")

### 1-2. Split

In [ ]:
# 텍스트 청크 분할 실행
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
)

chunks = text_splitter.split_documents(docs)
print(f"생성된 청크 수: {len(chunks)}")

### 1-3. 임베딩 Store

In [ ]:
USE_OPENAI_EMBEDDING = True

if MODEL_TYPE == "ollama":
    print("Ollama bge-m3 임베딩 모델 로딩...")
    embedding_model = OllamaEmbeddings(model="bge-m3")
elif USE_OPENAI_EMBEDDING:
    print("OpenAI 임베딩 모델 로딩...")
    os.environ["OPENAI_API_KEY"] = OPENAI_APIKEY
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
else:
    print("한국어 임베딩 무료 모델 로딩...")
    embedding_model = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )

In [ ]:
# ChromaDB 생성 및 벡터 저장
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=PERSIST_DIR
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": K_RETRIEVAL}
)

# 2. 정보 검색 및 텍스트 생성

In [ ]:
TEMPERATURE = 0.2

if MODEL_TYPE == "gpt":
    os.environ["OPENAI_API_KEY"] = OPENAI_APIKEY
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=TEMPERATURE)
elif MODEL_TYPE == "gguf":
    model_path = os.path.join(MODEL_DIR, "llama-model.gguf")
    llm = LlamaCpp(model_path=model_path, n_ctx=4096, n_threads=2, temperature=TEMPERATURE)
elif MODEL_TYPE == "ollama":
    llm = Ollama(model=OLLAMA_MODEL, temperature=TEMPERATURE)

In [ ]:
# RAG용 프롬프트 템플릿 정의
PROMPT_TEMPLATE = """
다음 컨텍스트만을 사용해서 한국어로 간결하고 정확하게 답변하세요.
컨텍스트에 없는 내용은 모른다고 답변하고, 추측하지 마세요.
가능하면 출처 정보(페이지 번호)도 함께 제공하세요.

컨텍스트:
{context}

질문: {question}
답변:
"""

prompt_template = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

In [ ]:
# RAG 체인 구성
def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs, 1):
        content = doc.page_content
        page = doc.metadata.get('page', '알 수 없음')
        formatted.append(f"[출처 {i}] 페이지 {page}\n{content}")
    return "\n\n".join(formatted)

retrieval_chain = RunnableParallel(
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
        "source_documents": retriever
    }
)

answer_chain = (
    {
        "context": lambda x: x["context"],
        "question": lambda x: x["question"]
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

rag_chain = retrieval_chain | RunnablePassthrough.assign(answer=answer_chain)

### 질의 테스트

In [ ]:
question = "스마트금융과 모집 일정은?"
full_result = rag_chain.invoke(question)
print(f"\n질문: {question}")
print(f"\n답변:\n{full_result['answer']}")

### FAST API 연동

In [ ]:
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
import nest_asyncio

app = FastAPI(title="ML API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["GET", "POST", "PUT", "DELETE"],
    allow_headers=["*"],
)

class InDataset(BaseModel):
    question : str

@app.post("/predict", status_code=200)
async def predict_tf(x: InDataset):
    response = rag_chain.invoke(x.question)
    return {"prediction": response['answer'], "references": response['source_documents'] }

@app.get('/')
async def root():
    return {"message": "online"}

In [ ]:
from pyngrok import ngrok

auth_token = os.getenv("NGROK_AUTH_TOKEN")
if auth_token:
    ngrok.set_auth_token(auth_token)
    ngrokTunnel = ngrok.connect(8000)
    print("공용 URL", ngrokTunnel.public_url)

nest_asyncio.apply()
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)